# B4b — Held-out (and dev100) caching for the B3 improvement candidates

## Purpose
Cache, for the **1,186 held-out FSC-147 val images**, every input the five frozen candidates need. No counts and no GT are computed here — only masks and maps — so this does not touch the pre-declared held-out comparison.

| Candidate | Needs |
|---|---|
| GPV_R4+P[MixA] | tiled maps, LM0 guided masks, Z-LM0 automatic masks |
| GPV_K-n+P[MixA] | + normals |
| GPV_K-n+P[LM0] | tiled maps, LM0 guided + LM0 automatic masks, normals |
| H1b+P[MixA] | tiled maps, LM0 guided, Z-LM0 automatic (A3c, depth already cached) |
| H1b-K3n+P[LM0] | tiled maps, LM0 guided + automatic, normals |

## Jobs (set `JOB` in Step 0.1; one job per Colab session)
| JOB | Content | Depends on | Time (T4) |
|---|---|---|---|
| **A** | tiled expanded-vocabulary SAN maps + Marigold normals | — | ~1–1.5 h (incl. installs) |
| **B** | LM0 guided masks + LM0 automatic masks (SAM2.1-Large, masked M0) | A complete | ~3 h |
| **C1** / **C2** | Z-LM0 zoomed automatic masks (SAM2.1-Large, masked M0, `crop_n_layers` 1), first / second half | A complete | ~6 h each |
| **D** | **dev100** (100 images): all of A, B and C in one session → `cache/FSC147_val_dev100_candidates/` — for the secondary full-validation report (held-out + dev100), as in B4 | — | ~1.5–2 h |

Run **A first**; when it reports 1,186/1,186, start **B, C1 and C2** in parallel sessions. **D** is independent and can run in any free session. Every job is resumable.

For D, if A.2 reports a protobuf mismatch: restart, re-run 0.1, 0.2, then continue from A.2.

All settings are exactly those used on train_dev400 (B3S §5, B3MN §2, B3L §5, B3Z): same vocabulary, tiles, normals model, SAM2 checkpoint/config, masking and discard rule.

In [1]:
# ============================================================
# Step 0.1 — Choose the job and mount Drive
# ============================================================

JOB = "B"        # held-out: "A", "B", "C1", "C2"  |  dev100 (all steps, one session): "D"
assert JOB in ("A", "B", "C1", "C2", "D")
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# ============================================================
# Step 0.2 — Paths, repo, held-out list
# ============================================================

import importlib, json, subprocess, sys, time, shutil
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

PROJECT_ROOT = Path("/content/drive/MyDrive/MSc_Project_ZeroShot_Counting")
FSC147_IMAGE_DIR = PROJECT_ROOT / "datasets" / "FSC147" / "raw" / "images_384_VarV2"
HO_ROOT = PROJECT_ROOT / "cache" / "FSC147_val_heldout_1186"
REPO_URL = "https://github.com/paulmcchan/Surrey-MScProject-Zero-Shot-Object-Counting.git"
REPO_DIR = Path("/content/Surrey-MScProject-Zero-Shot-Object-Counting")
!rm -rf "{REPO_DIR}"
!git clone -q "{REPO_URL}" "{REPO_DIR}"
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
REPO_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
import src.pipeline.io_cache as io_cache
import src.pipeline.b0 as b0
for m in (io_cache, b0):
    importlib.reload(m)

RESULTS_ROOT = PROJECT_ROOT / "results"
if JOB == "D":      # dev100: needed only for the secondary full-validation report (held-out + dev100)
    HO_ROOT = PROJECT_ROOT / "cache" / "FSC147_val_dev100_candidates"
    HO_ROOT.mkdir(parents=True, exist_ok=True)
    ho = pd.read_csv(RESULTS_ROOT / "B1_experiments" / "subsets" / "FSC147_val_dev_100.csv") \
           .sort_values("dataset_index").reset_index(drop=True)
else:
    ho = pd.read_csv(HO_ROOT / "FSC147_val_heldout_1186.csv").sort_values("dataset_index").reset_index(drop=True)
N_IMG = len(ho)
assert N_IMG == (100 if JOB == "D" else 1186), N_IMG
if "category" not in ho.columns:          # class names from the FSC-147 val split (no GT counts used)
    import src.datasets as datasets_mod
    _ds = datasets_mod.FSC147Dataset(PROJECT_ROOT / "datasets" / "FSC147" / "raw", split="val", load_images=False)
    ho["category"] = ho["image_id"].map(_ds.image_to_class)
assert ho["category"].notna().all()
TILED_DIR = HO_ROOT / "san_vocab_tiled"
NORMALS_DIR = HO_ROOT / "normals"
LM_ROOT = HO_ROOT / "sam2_large_masked_M0"
ZLM_ROOT = HO_ROOT / "zoomed" / "LM0"
def stem(iid): return io_cache.image_stem(iid)
def prompts_for(di, iid):
    if JOB == "D":   # dev100 dense_recovery prompts (frozen checkpoint cache)
        return io_cache.load_guided_masks(io_cache.guided_cache_path("dense_recovery", di, iid))["prompts_xy"]
    return io_cache.load_guided_masks(HO_ROOT / "guided_dense_recovery" / f"{int(di):04d}_{stem(iid)}.npz")["prompts_xy"]
print("JOB", JOB, "| repo", REPO_COMMIT[:10], "| images:", N_IMG, "| cache root:", HO_ROOT)

JOB B | repo d70bb1d6d6 | images: 1186 | cache root: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/cache/FSC147_val_heldout_1186


# Job A — tiled expanded-vocabulary SAN maps and normals

In [3]:
# ============================================================
# Step A.1 — Install SAN + detectron2 and diffusers (JOB A only)
# ============================================================

if JOB in ("A", "D"):
    import torch
    assert torch.cuda.is_available(), "Switch the runtime to T4."
    DETECTRON2_COMMIT = "1e3e13bbf607b54f62205c4c33922521822fb298"
    %cd /content
    !rm -rf /content/SAN
    !git clone -q https://github.com/MendelXu/SAN.git /content/SAN
    !pip install -q cython scipy shapely timm h5py submitit scikit-image wandb setuptools fvcore tabulate regex opencv-python "open_clip_torch==2.16.0"
    !pip uninstall -y -q detectron2
    !python -m pip install -q "git+https://github.com/facebookresearch/detectron2.git@{DETECTRON2_COMMIT}"
    !pip install -q "accelerate==1.14.0" "diffusers==0.40.0" "transformers==5.16.1"
    !pip install -q "protobuf==3.20.3" "iopath==0.1.10"
    r = subprocess.run([sys.executable, "-c",
        "import sys; sys.path[:0]=['/content/SAN']; import google.protobuf, detectron2, san; "
        "from diffusers import MarigoldNormalsPipeline; print('IMPORTS OK', google.protobuf.__version__)"],
        cwd="/content", capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr[-2000:])
    assert r.returncode == 0
    print("If A.2 reports a protobuf mismatch: restart the session, re-run 0.1, 0.2, then A.2.")
else:
    print("Skipped (JOB is", JOB, ")")

Skipped (JOB is B )


In [4]:
# ============================================================
# Step A.2 — Tiled SAN maps for held-out (JOB A; resumable)
# ============================================================
# B3S Step 5.1 settings: vocabulary = [class, background] + 43 stuff terms
# (overlaps removed); 2 x 2 corner-anchored tiles, 60% of each side;
# class scores averaged over overlaps; saved: tiled argmax.

if JOB in ("A", "D"):
    import google.protobuf, importlib.metadata as md_meta
    assert md_meta.version("protobuf") == "3.20.3" and google.protobuf.__version__ == "3.20.3", \
        "protobuf mismatch — restart, re-run 0.1, 0.2, then A.2."
    if "/content/SAN" not in sys.path:
        sys.path.insert(0, "/content/SAN")
    import torch
    from PIL import Image
    import src.pipeline.semantic as semantic
    importlib.reload(semantic)
    DEVICE = "cuda"
    SAN_ROOT = Path("/content/SAN")
    SAN_CKPT = SAN_ROOT / "output" / "san_vit_b_16.pth"
    SAN_CKPT.parent.mkdir(parents=True, exist_ok=True)
    if not SAN_CKPT.exists():
        shutil.copy2(PROJECT_ROOT / "Checkpoints" / "SAN" / "san_vit_b_16.pth", SAN_CKPT)
    san_model = semantic.build_san(SAN_ROOT, SAN_CKPT, device=DEVICE)
    TILE_FRAC = 0.60

    def tiled_scores(img, vocab):
        w, h = img.size
        th, tw = int(np.ceil(TILE_FRAC * h)), int(np.ceil(TILE_FRAC * w))
        acc = np.zeros((len(vocab), h, w), np.float32); cnt = np.zeros((h, w), np.float32)
        for y0 in (0, h - th):
            for x0 in (0, w - tw):
                seg = semantic.run_san_vocab(san_model, img.crop((x0, y0, x0 + tw, y0 + th)), vocab, DEVICE)
                acc[:, y0:y0 + th, x0:x0 + tw] += seg; cnt[y0:y0 + th, x0:x0 + tw] += 1.0
        return acc / cnt[None]

    TILED_DIR.mkdir(parents=True, exist_ok=True)
    todo = [r for r in ho.itertuples(index=False) if not (TILED_DIR / f"{stem(r.image_id)}.npz").exists()]
    print(f"Tiled maps — done: {len(ho) - len(todo)} | to process: {len(todo)}")
    t0 = time.time()
    for r in tqdm(todo, desc="Tiled SAN (held-out)"):
        img = Image.open(FSC147_IMAGE_DIR / r.image_id).convert("RGB")
        vocab = semantic.expanded_vocabulary(r.category)
        seg = tiled_scores(img, vocab)
        np.savez_compressed(TILED_DIR / f"{stem(r.image_id)}.npz",
                            tiled_argmax=seg.argmax(axis=0).astype(np.uint8), vocabulary=np.array(vocab))
    print(f"Elapsed: {(time.time() - t0) / 60:.1f} min")
else:
    print("Skipped (JOB is", JOB, ")")

Skipped (JOB is B )


In [5]:
# ============================================================
# Step A.3 — Marigold normals for held-out (JOB A; resumable)
# ============================================================
# B3MN settings: prs-eth/marigold-normals-v1-1, fp16, 4 steps, ensemble 1, 768 px, seed 0.

if JOB in ("A", "D"):
    import torch
    from PIL import Image
    from diffusers import MarigoldNormalsPipeline
    pipe_n = MarigoldNormalsPipeline.from_pretrained("prs-eth/marigold-normals-v1-1", dtype=torch.float16).to("cuda")
    pipe_n.set_progress_bar_config(disable=True)
    NORMALS_DIR.mkdir(parents=True, exist_ok=True)
    todo = [r for r in ho.itertuples(index=False) if not (NORMALS_DIR / f"{stem(r.image_id)}.npz").exists()]
    print(f"Normals — done: {len(ho) - len(todo)} | to process: {len(todo)}")
    t0 = time.time()
    for r in tqdm(todo, desc="Normals (held-out)"):
        img = Image.open(FSC147_IMAGE_DIR / r.image_id).convert("RGB")
        g = torch.Generator(device="cuda").manual_seed(0)
        with torch.inference_mode():
            out = pipe_n(img, num_inference_steps=4, ensemble_size=1, processing_resolution=768, generator=g)
        nrm = np.squeeze(np.asarray(out.prediction)).astype(np.float32)
        assert nrm.shape[:2] == (img.size[1], img.size[0]) and nrm.shape[2] == 3, nrm.shape
        np.savez_compressed(NORMALS_DIR / f"{stem(r.image_id)}.npz", normals=nrm.astype(np.float16))
    print(f"Elapsed: {(time.time() - t0) / 60:.1f} min")
    n_t = len(list(TILED_DIR.glob("*.npz"))); n_n = len(list(NORMALS_DIR.glob("*.npz")))
    print(f"JOB {JOB} — tiled maps {n_t}/{N_IMG} | normals {n_n}/{N_IMG}",
          ("→ COMPLETE" + (": start B, C1, C2" if JOB == "A" else "")) if n_t == n_n == N_IMG else "→ incomplete, re-run A.2/A.3")
else:
    print("Skipped (JOB is", JOB, ")")

Skipped (JOB is B )


# Jobs B / C — SAM2.1-Large masks on the masked input

In [6]:
# ============================================================
# Step S.1 — Install SAM2 and build SAM2.1-Large (JOBS B, C1, C2)
# ============================================================

if JOB in ("B", "C1", "C2", "D"):
    import torch
    assert torch.cuda.is_available(), "Switch the runtime to T4."
    n_t = len(list(TILED_DIR.glob("*.npz"))) if TILED_DIR.exists() else 0
    assert n_t == N_IMG, f"Tiled maps not complete: {n_t}/{N_IMG}"
    DEVICE = "cuda"
    %cd /content
    if not Path("/content/sam2_repo/sam2/__init__.py").exists():
        !git clone -q https://github.com/facebookresearch/sam2.git /content/sam2_repo
        !git -C /content/sam2_repo checkout -q {b0.FROZEN_SAM2_COMMIT}
        %cd /content/sam2_repo
        !pip install -q -e .
        %cd /content
    if "/content/sam2_repo" not in sys.path:
        sys.path.insert(0, "/content/sam2_repo")
    importlib.invalidate_caches()
    from sam2.build_sam import build_sam2
    from sam2.sam2_image_predictor import SAM2ImagePredictor
    from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator
    SAM2_L = PROJECT_ROOT / "Checkpoints" / "SAM2" / "sam2.1_hiera_large.pt"
    CFG_L = "configs/sam2.1/sam2.1_hiera_l.yaml"
    assert SAM2_L.exists()
    SAM2_L_SHA = b0.sha256_file(SAM2_L)
    if JOB in ("B", "D"):
        predictor_L = SAM2ImagePredictor(build_sam2(CFG_L, str(SAM2_L), device=DEVICE).eval())
        amg_L = b0.build_amg(SAM2_L, device=DEVICE, model_cfg=CFG_L, verify_sha256=False)
    if JOB in ("C1", "C2", "D"):
        model_L = build_sam2(CFG_L, str(SAM2_L), device=DEVICE, apply_postprocessing=False).eval()
        amg_L_Z = SAM2AutomaticMaskGenerator(model_L, points_per_side=32, crop_n_layers=1)
    print("SAM2.1-Large ready | SHA256", SAM2_L_SHA[:16], "| GPU", torch.cuda.get_device_name(0))

    def masked_image(r):
        img = b0.load_image_rgb_cv2(FSC147_IMAGE_DIR / r.image_id)
        with np.load(TILED_DIR / f"{stem(r.image_id)}.npz") as v:
            reg = v["tiled_argmax"] == 0
        return img, reg, (img * reg[..., None]).astype(np.uint8)

    def keep_inside(masks, region):
        if masks.shape[0] == 0:
            return np.zeros(0, bool)
        inside = (masks & region[None]).reshape(masks.shape[0], -1).sum(1)
        return inside > 0.5 * np.maximum(masks.reshape(masks.shape[0], -1).sum(1), 1)

    def filter_meta(meta, keep):
        return {k: np.asarray(v)[keep] for k, v in meta.items()}

    def safe_amg(amg, image):
        try:
            return b0.run_amg(amg, image), False
        except IndexError:            # SAM2 empty-result case with crops
            h, w = image.shape[:2]
            return (np.zeros((0, h, w), bool), {"areas": np.zeros(0, np.int64), "predicted_iou": np.zeros(0, np.float32),
                    "stability_score": np.zeros(0, np.float32), "bbox_xywh": np.zeros((0, 4)),
                    "crop_box_xywh": np.zeros((0, 4)), "point_xy": np.zeros((0, 2))}), True
else:
    print("Skipped (JOB is", JOB, ")")

/content
SAM2.1-Large ready | SHA256 2647878d5dfa5098 | GPU NVIDIA L4


In [7]:
# ============================================================
# Step B.1 — LM0 guided + automatic masks (JOB B; resumable)
# ============================================================
# B3L §5: SAM2.1-Large on the masked image (target-winning tiled region, no dilation);
# masks with > 50% of pixels outside the region are discarded.

if JOB in ("B", "D"):
    for d in ["b0", "guided", "_done"]:
        (LM_ROOT / d).mkdir(parents=True, exist_ok=True)
    todo = [r for r in ho.itertuples(index=False) if not (LM_ROOT / "_done" / f"{int(r.dataset_index):05d}.json").exists()]
    print(f"LM0 — done: {len(ho) - len(todo)} | to process: {len(todo)}")
    t0 = time.time()
    for i, r in enumerate(tqdm(todo, desc="LM0 (held-out)")):
        di, s = r.dataset_index, stem(r.image_id)
        img, reg, masked = masked_image(r)
        (bm, bmeta), empty = safe_amg(amg_L, masked)
        kb = keep_inside(bm, reg)
        io_cache.save_b0_masks(LM_ROOT / "b0" / f"{int(di):05d}_{s}.npz", bm[kb], filter_meta(bmeta, kb))
        prompts = prompts_for(di, r.image_id)
        h, w = img.shape[:2]
        gm, gs = [], []
        if len(prompts):
            with torch.inference_mode():
                predictor_L.set_image(masked)
                for p in prompts:
                    m, sc, _ = predictor_L.predict(point_coords=np.asarray([p], np.float32),
                                                   point_labels=np.asarray([1], np.int32), multimask_output=False)
                    gm.append(np.asarray(m[0], bool)); gs.append(float(np.asarray(sc)[0]))
        gm = np.asarray(gm) if gm else np.zeros((0, h, w), bool); gs = np.asarray(gs, float)
        kg = keep_inside(gm, reg)
        io_cache.save_guided_masks(LM_ROOT / "guided" / f"{int(di):04d}_{s}.npz", gm[kg],
                                   prompts[kg] if len(prompts) else prompts, gs[kg], (h, w),
                                   dataset_index=di, image_id=r.image_id)
        with open(LM_ROOT / "_done" / f"{int(di):05d}.json", "w") as f:
            json.dump({"dataset_index": int(di), "b0_kept": int(kb.sum()), "guided_kept": int(kg.sum()),
                       "amg_empty_fallback": bool(empty)}, f)
        if (i + 1) % 25 == 0:
            torch.cuda.empty_cache()
    print(f"Elapsed: {(time.time() - t0) / 60:.1f} min")
    print(f"JOB {JOB} LM0 status —", len(list((LM_ROOT / "_done").glob("*.json"))), f"/ {N_IMG}")
else:
    print("Skipped (JOB is", JOB, ")")

LM0 — done: 0 | to process: 1186


LM0 (held-out):   0%|          | 0/1186 [00:00<?, ?it/s]

Elapsed: 98.3 min
JOB B LM0 status — 1186 / 1186


In [8]:
# ============================================================
# Step C.1 — Z-LM0 zoomed automatic masks (JOBS C1 / C2; resumable)
# ============================================================
# B3Z [LM0]: SAM2.1-Large AMG, points_per_side 32, crop_n_layers 1,
# on the masked image; masks > 50% outside the region discarded.
# C1 = first half of the held-out list (by dataset_index), C2 = second half.

if JOB in ("C1", "C2", "D"):
    half = len(ho) // 2
    part = ho if JOB == "D" else (ho.iloc[:half] if JOB == "C1" else ho.iloc[half:])
    for d in ["b0", "_done"]:
        (ZLM_ROOT / d).mkdir(parents=True, exist_ok=True)
    todo = [r for r in part.itertuples(index=False) if not (ZLM_ROOT / "_done" / f"{int(r.dataset_index):05d}.json").exists()]
    print(f"Z-LM0 {JOB} — part size {len(part)} | done: {len(part) - len(todo)} | to process: {len(todo)}")
    t0 = time.time()
    for i, r in enumerate(tqdm(todo, desc=f"Z-LM0 {JOB} (held-out)")):
        di, s = r.dataset_index, stem(r.image_id)
        img, reg, masked = masked_image(r)
        (bm, bmeta), empty = safe_amg(amg_L_Z, masked)
        kb = keep_inside(bm, reg)
        io_cache.save_b0_masks(ZLM_ROOT / "b0" / f"{int(di):05d}_{s}.npz", bm[kb], filter_meta(bmeta, kb))
        with open(ZLM_ROOT / "_done" / f"{int(di):05d}.json", "w") as f:
            json.dump({"dataset_index": int(di), "b0_kept": int(kb.sum()), "amg_empty_fallback": bool(empty)}, f)
        if (i + 1) % 20 == 0:
            torch.cuda.empty_cache()
    print(f"Elapsed: {(time.time() - t0) / 60:.1f} min")
    print("JOB", JOB, "status — Z-LM0 total", len(list((ZLM_ROOT / "_done").glob("*.json"))), f"/ {N_IMG}")
else:
    print("Skipped (JOB is", JOB, ")")

Skipped (JOB is B )


# Final check (any session, once all jobs are done)

In [9]:
# ============================================================
# Step Z.1 — Completeness check and metadata (any runtime)
# ============================================================

def count(p, pat="*"):
    return len(list(p.glob(pat))) if p.exists() else 0
status = {"tiled_maps": count(TILED_DIR, "*.npz"), "normals": count(NORMALS_DIR, "*.npz"),
          "LM0_done": count(LM_ROOT / "_done", "*.json"), "Z_LM0_done": count(ZLM_ROOT / "_done", "*.json")}
empty_lm = sum(json.load(open(p)).get("amg_empty_fallback", False) for p in (LM_ROOT / "_done").glob("*.json")) if LM_ROOT.exists() else 0
empty_z = sum(json.load(open(p)).get("amg_empty_fallback", False) for p in (ZLM_ROOT / "_done").glob("*.json")) if ZLM_ROOT.exists() else 0
print(status, "| AMG empty-result fallbacks: LM0", empty_lm, "| Z-LM0", empty_z)
ALL_DONE = all(v == N_IMG for v in status.values())
print("CACHE (" + ("dev100" if JOB == "D" else "held-out") + "):", "COMPLETE" if ALL_DONE else "INCOMPLETE")
if ALL_DONE:
    with open(HO_ROOT / "B4b_candidate_cache_metadata.json", "w") as f:
        json.dump({"notebook": "B4b_HeldOut_Caching_Candidates", "subset": "dev100" if JOB == "D" else "heldout",
                   "images": N_IMG, **status,
                   "amg_empty_fallbacks": {"LM0": int(empty_lm), "Z_LM0": int(empty_z)},
                   "settings": {"tiled": "B3S s5.1 (2x2, 60%, averaged; expanded vocabulary)",
                                "normals": "marigold-normals-v1-1, fp16, 4 steps, ens 1, 768, seed 0",
                                "LM0": "SAM2.1-Large, masked M0, discard >50% outside; guided = dense_recovery prompts",
                                "Z_LM0": "SAM2.1-Large AMG pps 32, crop_n_layers 1, masked M0, discard >50% outside"},
                   "repo_commit": REPO_COMMIT, "counts_computed": False, "gt_accessed": False,
                   "test_set_accessed": False,
                   "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")}, f, indent=2)
    print("Metadata saved.")

{'tiled_maps': 1186, 'normals': 1186, 'LM0_done': 1186, 'Z_LM0_done': 1186} | AMG empty-result fallbacks: LM0 0 | Z-LM0 1
CACHE (held-out): COMPLETE
Metadata saved.


In [10]:
# ============================================================
# Step Z.2 — Full completeness + readability check (held-out AND dev100)
# ============================================================
# Opens every cached file with the loader the evaluation will use.
# Set DELETE_BAD = True to remove unreadable files so the relevant job regenerates them.

DELETE_BAD = False

SUBSETS = {
    "held-out": (PROJECT_ROOT / "cache" / "FSC147_val_heldout_1186",
                 pd.read_csv(PROJECT_ROOT / "cache" / "FSC147_val_heldout_1186" / "FSC147_val_heldout_1186.csv")),
    "dev100":   (PROJECT_ROOT / "cache" / "FSC147_val_dev100_candidates",
                 pd.read_csv(PROJECT_ROOT / "results" / "B1_experiments" / "subsets" / "FSC147_val_dev_100.csv")),
}

def check_file(path, kind):
    try:
        if kind == "tiled":
            with np.load(path) as v: _ = v["tiled_argmax"].shape
        elif kind == "normals":
            with np.load(path) as v: _ = v["normals"].shape
        elif kind == "b0":
            io_cache.load_b0_masks(path)
        elif kind == "guided":
            io_cache.load_guided_masks(path)
        return None
    except Exception as e:
        return repr(e)

summary, all_bad = [], []
t0 = time.time()
for name, (root, lst) in SUBSETS.items():
    expected = {
        "tiled maps":          [(root / "san_vocab_tiled" / f"{stem(r.image_id)}.npz", "tiled") for r in lst.itertuples()],
        "normals":             [(root / "normals" / f"{stem(r.image_id)}.npz", "normals") for r in lst.itertuples()],
        "LM0 automatic":       [(root / "sam2_large_masked_M0" / "b0" / f"{int(r.dataset_index):05d}_{stem(r.image_id)}.npz", "b0") for r in lst.itertuples()],
        "LM0 guided":          [(root / "sam2_large_masked_M0" / "guided" / f"{int(r.dataset_index):04d}_{stem(r.image_id)}.npz", "guided") for r in lst.itertuples()],
        "Z-LM0 automatic":     [(root / "zoomed" / "LM0" / "b0" / f"{int(r.dataset_index):05d}_{stem(r.image_id)}.npz", "b0") for r in lst.itertuples()],
    }
    for cache, items in expected.items():
        missing = [p for p, _ in items if not p.exists()]
        bad = []
        for p, kind in tqdm([(p, k) for p, k in items if p.exists()], desc=f"{name} | {cache}", leave=False):
            err = check_file(p, kind)
            if err:
                bad.append(p); all_bad.append((name, cache, p.name, err))
        summary.append({"subset": name, "cache": cache, "expected": len(items),
                        "present": len(items) - len(missing), "readable": len(items) - len(missing) - len(bad),
                        "missing": len(missing), "unreadable": len(bad),
                        "status": "OK" if not missing and not bad else "PROBLEM"})
        if DELETE_BAD:
            for p in bad:
                p.unlink()

print(f"Checked in {(time.time() - t0) / 60:.1f} min")
display(pd.DataFrame(summary))
for name, cache, fname, err in all_bad:
    print("UNREADABLE:", name, "|", cache, "|", fname, "|", err)
ALL_OK = all(s["status"] == "OK" for s in summary)
print("ALL CACHES (held-out + dev100):", "COMPLETE AND READABLE" if ALL_OK else "PROBLEMS FOUND — see table")

held-out | tiled maps:   0%|          | 0/1186 [00:00<?, ?it/s]

held-out | normals:   0%|          | 0/1186 [00:00<?, ?it/s]

held-out | LM0 automatic:   0%|          | 0/1186 [00:00<?, ?it/s]

held-out | LM0 guided:   0%|          | 0/1186 [00:00<?, ?it/s]

held-out | Z-LM0 automatic:   0%|          | 0/1186 [00:00<?, ?it/s]

...

dev100 | LM0 guided:   0%|          | 0/100 [00:00<?, ?it/s]

dev100 | Z-LM0 automatic:   0%|          | 0/100 [00:00<?, ?it/s]

Checked in 23.4 min


,subset,cache,expected,present,readable,missing,unreadable,status
0,held-out,tiled maps,1186,1186,1186,0,0,OK
1,held-out,normals,1186,1186,1186,0,0,OK
2,held-out,LM0 automatic,1186,1186,1186,0,0,OK
3,held-out,LM0 guided,1186,1186,1186,0,0,OK
4,held-out,Z-LM0 automatic,1186,1186,1186,0,0,OK
5,dev100,tiled maps,100,100,100,0,0,OK
6,dev100,normals,100,100,100,0,0,OK
7,dev100,LM0 automatic,100,100,100,0,0,OK
8,dev100,LM0 guided,100,100,100,0,0,OK
9,dev100,Z-LM0 automatic,100,100,100,0,0,OK


ALL CACHES (held-out + dev100): COMPLETE AND READABLE


# B4b conclusion

**Caching complete and verified** for the five B3 improvement candidates (GPV_R4+P[MixA], GPV_K-n+P[MixA], GPV_K-n+P[LM0], H1b+P[MixA], H1b-K3n+P[LM0]). No counts and no GT were computed — the pre-declared held-out comparison is untouched.

**Run record** (one notebook, five copies differing only in `JOB`; all on NVIDIA **L4**):

| Job | Content | Images | Approx. time |
|---|---|---|---|
| A | held-out tiled expanded-vocabulary SAN maps + Marigold normals | 1,186 | ~1–1.5 h |
| C1 | held-out Z-LM0 zoomed automatic masks, first half | 593 | ~2.5 h (interrupted at 536; resumed) |
| C2 | held-out Z-LM0 zoomed automatic masks, second half | 593 | ~2.6 h |
| B | held-out LM0 guided + automatic masks | 1,186 | ~1.6 h |
| D | dev100: tiled maps, normals, LM0 and Z-LM0 masks | 100 | ~1.5 h |

**Incident:** one held-out tiled map (`4137.npz`) was written incompletely during job A (truncated on Drive) and stopped job C1 at image 537. A readability scan found it was the only bad file; it was deleted and regenerated by re-running job A, and C1 resumed.

**SAM2 empty-result cases** (handled as "no masks"): held-out LM0 0, Z-LM0 1; dev100 0.

**Checks:**
- Z.1 (counts): held-out 1,186/1,186 and dev100 100/100 for all four cache types.
- Z.2 (every file opened with the evaluation loaders): all 10 caches **complete and readable**.

**GPU note:** train-pool caches were made on T4, these on L4; GPU-level floating-point differences are expected to be negligible (cf. B4 drift test: 5,743/5,744 SAM2 prompts identical, all GPV_R4 counts identical). All candidates are compared on the same L4 caches.

**Next:** freeze the candidate code in `src/`, pre-declare the held-out decision rule, then run the held-out (primary) and full-validation (secondary) evaluation with B0 and H1b_v1 as references.